# CICIoT2023 — Holdout blending DT + RF + XGBoost (nhánh duong)

**8 lớp; preprocessing cũ; base-model cố định; meta-model học từ 24 xác suất trên val.**
Notebook triển khai `work_dir/Nguyen_Ngoc_Duong_04_ke_hoach_blending_meta_model.md`.
Chạy **Kaggle CPU**, không chạy training/inference trên máy local. Bản notebook này được tạo trên `duong`, chưa commit.

Val được dùng làm **meta-development/training**, chọn meta bằng CV 3 folds theo group raw-float32.
Đây là holdout blending; không phải soft voting hay OOF stacking của các base-model.
Base configs đã được chọn theo val trước đây; baseline val/test và các kết quả recall/XAI đã được xem.
OOF dưới đây là điểm development có selection bias; test cuối là **test lịch sử**, không phải test chưa từng xem.
Blending học thêm nhãn val nên ngân sách dữ liệu khác classifier đơn chỉ học train.

**Luồng mặc định:** tái sử dụng base đã fit → audit → Z_val → 18 pilot fits → tối đa 24 tuning fits
→ fit meta toàn val → export/reload → freeze → dừng trước test.
Không fit lại base trên train+val. Không dùng test để chọn cấu hình, weight hay ngưỡng.
Không bảo đảm blending sẽ tăng recall; cả kết quả âm cũng cần được báo cáo.


## 0. Chuẩn bị input, môi trường và cách chạy

Attach **toàn bộ các run** (giữ cấu trúc thư mục), không chỉ thư mục handoff:

| Đầu vào | Notebook nguồn | Run |
|---|---|---|
| Processed raw arrays + labels + metadata/config | `ciciot2023-do-an-processing.ipynb` | `20261007_164739_35f682b9` |
| Pilot 1M + manifest/indices | `ciciot2023-do-an-pilot.ipynb` | `pilot_20261008_022915_ea9a88b0` |
| 3 fitted bundles, caches, predictions, audits/masks/checksums | `ciciot2023-do-an-models.ipynb` | `models_20261008_073827_d29a37c3` |

Không dùng output `recall_experiments` thay baseline. Notebook này **reuse** base đã huấn luyện;
không có phase refit base. Thiếu cache thì attach đầy đủ run baseline trước khi tiếp tục.

Môi trường phải tương thích model đã serialize. Cell kiểm tra version sẽ dừng trước `joblib.load` nếu lệch.
Nếu cần cài lại trên Kaggle, mở Internet và dùng cell riêng, sau đó **Restart kernel**:

```python
%pip install numpy==2.1.3 pandas==2.3.3 scipy==1.16.3 scikit-learn==1.6.1 xgboost==3.4.1 joblib==1.6.0
```

Python cần cùng major/minor với baseline (3.13). Không tự bỏ kiểm tra version/golden.
Nếu package/Python này không còn khả dụng, cần một run baseline mới trong môi trường thống nhất.

- Lần đầu: `RESUME_FROM=None`, `RUN_FINAL_TEST=False`, **Restart → Run All**.
- Hết ngân sách/Stop: **Save Version** giữ output, attach output version đó ở lần sau;
  đặt `RESUME_FROM` tới **thư mục run chứa `protocol.json`**, không trỏ tới một file JSON.
- Sau khi xem CV và freeze: dùng lại run đó, đặt `RUN_FINAL_TEST=True`, chạy lại.
  Tăng `SEARCH_BUDGET_HOURS` để tiếp tục tìm kiếm còn dang dở; các thay đổi protocol khác cần run mới.
- Kaggle bắt buộc dừng session trước khi Save Version nếu đang chạy; fold đang chạy có thể phải làm lại,
  nhưng các fold đã có completion marker được giữ. Không xóa marker để chạy chọn lại theo test.


In [ ]:
from pathlib import Path
import os
# Cấu hình Kaggle. Có thể để None để tự tìm; nếu nhiều run cùng tên, điền đường dẫn cụ thể.
PROCESSED_DIR = None
PILOT_DIR = None
BASELINE_DIR = None
RESUME_FROM = None
RUN_FINAL_TEST = False
SEARCH_BUDGET_HOURS = 4.0  # Tổng thời gian search đã checkpoint; tăng số này nếu cần resume.
RUN_SEED_STABILITY = False  # Nếu bật: cùng config/folds, seeds 2026 và 3407; không chọn seed theo test.
N_THREADS = 4
BATCH_SIZE = 25000
BENCHMARK_REPEATS = 30
TUNING_TRIALS = 8
MIN_RARE_PRECISION = 0.30
MACRO_F1_TOLERANCE = 0.002
N_FOLDS = 3
SEED = 42
for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[key] = str(N_THREADS)


In [ ]:
import gc
import hashlib
import importlib
import importlib.metadata
import json
import platform
import shutil
import sys
import threading
import time
import uuid
from datetime import datetime, timezone
import joblib
import numpy as np
import pandas as pd
import psutil
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterSampler, StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, log_loss
from sklearn.utils.validation import check_is_fitted
from threadpoolctl import threadpool_limits
from xgboost import XGBClassifier

assert Path('/kaggle/input').is_dir() and Path('/kaggle/working').is_dir(), 'Chỉ chạy notebook này trên Kaggle.'
CLASSES = ['Normal', 'BruteForce', 'DDoS', 'DoS', 'Mirai', 'Recon', 'Spoofing', 'Web-Based']
BASE_ORDER = ['DT', 'RF', 'XGBoost']
BASE_IDS = dict(DT='DT_n1000000_none_s42', RF='RF_n1000000_none_s42', XGBoost='XGBoost_n1000000_sqrt_s42')
SOURCE_RUN = '20261007_164739_35f682b9'
PILOT_RUN = 'pilot_20261008_022915_ea9a88b0'
BASE_RUN = 'models_20261008_073827_d29a37c3'
NOTEBOOK_REVISION = '1.0'
AUTHOR_BRANCH = 'duong'
AUTHOR_BASE_COMMIT = 'b144b4b8561421b921f29a29295f423d48727b0e'
TEMPLATE_SOURCE_SHA256 = 'e47f6dea2c4e07f185cd0b78e9cd3c3c88e172bb5fea322d35f842320bdb096e'
assert N_FOLDS == 3 and TUNING_TRIALS <= 8 and N_THREADS > 0
assert SEARCH_BUDGET_HOURS > 0 and BATCH_SIZE > 0 and BENCHMARK_REPEATS > 0


def now():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def canonical(obj):
    return json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(',', ':'), allow_nan=False)


def object_hash(obj):
    return hashlib.sha256(canonical(obj).encode()).hexdigest()


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def require_hash(path, expected):
    path = Path(path)
    assert path.is_file(), f'Thiếu input: {path}'
    actual = sha256(path)
    assert actual == expected, f'SHA256 không khớp: {path}'
    return actual


def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    tmp.replace(path)


def atomic_npy(path, array):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with tmp.open('wb') as stream:
        np.save(stream, array, allow_pickle=False)
    tmp.replace(path)


def atomic_joblib(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    joblib.dump(obj, tmp, compress=3)
    tmp.replace(path)


def seal(folder, marker='complete.json', extra=None):
    folder = Path(folder)
    hashes = {str(p.relative_to(folder)): sha256(p) for p in sorted(folder.rglob('*'))
              if p.is_file() and p.name != marker and not p.name.endswith('.tmp') and '__pycache__' not in p.parts}
    atomic_json(folder / marker, {'files': hashes, 'finished_at': now(), **(extra or {})})


def verify_seal(folder, marker='complete.json'):
    record = read_json(Path(folder) / marker)
    for relative, expected in record['files'].items():
        require_hash(Path(folder) / relative, expected)
    return record


def resolve_run(value, run_id, required):
    if value is not None:
        path = Path(value)
        assert all((path / f).is_file() for f in required), f'Input không đủ: {path}'
        return path
    matches = [p for p in Path('/kaggle/input').rglob(run_id)
               if p.is_dir() and all((p / f).is_file() for f in required)]
    # Upload có thể bỏ cấp run_id. Chỉ dùng fallback nếu tìm được duy nhất một thư mục phù hợp.
    if not matches:
        matches = sorted({p.parent for p in Path('/kaggle/input').rglob(required[0])
                          if all((p.parent / f).is_file() for f in required)})
    assert len(matches) == 1, f'{run_id}: tìm thấy {len(matches)} input; điền đường dẫn DIR cụ thể.'
    return matches[0]


PROCESSED_DIR = resolve_run(PROCESSED_DIR, SOURCE_RUN, ['preprocessing_config.json', 'X_val_raw.npy', 'y_val.npy'])
PILOT_DIR = resolve_run(PILOT_DIR, PILOT_RUN, ['pilot_manifest.json', 'train_1000000/pilot_manifest.json'])
BASELINE_DIR = resolve_run(BASELINE_DIR, BASE_RUN, ['protocol.json', 'artifact_checksums.json', 'handoff/DT/pipeline.joblib'])
BASE_PROTOCOL = read_json(BASELINE_DIR / 'protocol.json')
VERSIONS = {key: importlib.metadata.version(key) for key in
            ['numpy', 'pandas', 'scipy', 'scikit-learn', 'xgboost', 'joblib', 'psutil', 'matplotlib']}
VERSIONS['python'] = platform.python_version()
for key in ['numpy', 'pandas', 'scipy', 'scikit-learn', 'xgboost', 'joblib']:
    assert VERSIONS[key] == BASE_PROTOCOL['versions'][key], (
        f'{key}: runtime={VERSIONS[key]}, baseline={BASE_PROTOCOL["versions"][key]}. Cài version tương thích rồi restart.')
assert VERSIONS['python'].split('.')[:2] == BASE_PROTOCOL['versions']['python'].split('.')[:2], 'Python major/minor lệch baseline.'
print('Inputs:', PROCESSED_DIR, PILOT_DIR, BASELINE_DIR, sep='\n')
print('Versions:', VERSIONS)


## 1. Audit provenance và nạp đúng base-model đã fit trên train

Kiểm tra checksum, schema, train manifest và golden probabilities trước khi xây Z.
Dùng các audit/mask raw-float32 baseline đã lưu, không purge lại theo nhãn hay kết quả dự đoán.
Notebook chỉ đọc nội dung val ở giai đoạn này; test được chặn đến sau freeze.


In [ ]:
BASE_CHECKSUMS = read_json(BASELINE_DIR / 'artifact_checksums.json')
INPUT_INTEGRITY = {}


def baseline_file(relative):
    assert relative in BASE_CHECKSUMS, f'Input không có trong checksum manifest: {relative}'
    path = BASELINE_DIR / relative
    INPUT_INTEGRITY['baseline/' + relative] = require_hash(path, BASE_CHECKSUMS[relative])
    return path


assert BASE_PROTOCOL['source_run'] == SOURCE_RUN and BASE_PROTOCOL['pilot_run'] == PILOT_RUN
BASE_PROTOCOL_FILE_SHA256 = sha256(baseline_file('protocol.json'))
# Baseline dùng object_hash với JSON separators mặc định; khác checksum file.
BASE_PROTOCOL_HASH = hashlib.sha256(json.dumps(BASE_PROTOCOL, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
require_hash(PROCESSED_DIR / 'preprocessing_config.json', BASE_PROTOCOL['source_config_sha256'])
require_hash(PILOT_DIR / 'pilot_manifest.json', BASE_PROTOCOL['pilot_manifest_sha256'])
PROCESSING_CONFIG = read_json(PROCESSED_DIR / 'preprocessing_config.json')
PILOT_MANIFEST = read_json(PILOT_DIR / 'train_1000000/pilot_manifest.json')
FEATURES = PROCESSING_CONFIG['final_features']
assert len(FEATURES) == 44 and len(set(FEATURES)) == 44
assert PROCESSING_CONFIG['run_id'] == SOURCE_RUN and PROCESSING_CONFIG['class_names'] == CLASSES
assert PILOT_MANIFEST['feature_names'] == FEATURES and PILOT_MANIFEST['class_names'] == CLASSES
assert PILOT_MANIFEST['budget'] == 1000000
INPUT_INTEGRITY['processing_config'] = sha256(PROCESSED_DIR / 'preprocessing_config.json')
INPUT_INTEGRITY['pilot_manifest'] = sha256(PILOT_DIR / 'pilot_manifest.json')
INPUT_INTEGRITY['train_manifest'] = sha256(PILOT_DIR / 'train_1000000/pilot_manifest.json')
# Dữ liệu train không được fit lại. Kiểm tra manifest/targets/indices chứng minh nguồn training.
for relative in ['train_indices.npy', 'source_row_indices.npy', 'y_train.npy']:
    INPUT_INTEGRITY['pilot/' + relative] = require_hash(
        PILOT_DIR / 'train_1000000' / relative, PILOT_MANIFEST['checksums_sha256'][relative])
BASE_PIPELINES, BASE_SNAPSHOT = {}, {}
for name in BASE_ORDER:
    prefix = 'handoff/' + name + '/'
    config = read_json(baseline_file(prefix + 'config.json'))
    bundle = read_json(baseline_file(prefix + 'bundle_metadata.json'))
    assert config['experiment_id'] == BASE_IDS[name] and config['model'] == name
    assert config['features'] == FEATURES and config['class_order'] == CLASSES
    assert config['budget'] == 1000000 and config['protocol_hash'] == BASE_PROTOCOL_HASH
    assert config['source_run'] == SOURCE_RUN and config['pilot_run'] == PILOT_RUN
    assert config['train_manifest_sha256'] == INPUT_INTEGRITY['train_manifest']
    require_hash(baseline_file(prefix + 'model.joblib'), bundle['model_sha256'])
    pipeline_path = baseline_file(prefix + 'pipeline.joblib')
    require_hash(pipeline_path, bundle['pipeline_sha256'])
    pipeline = joblib.load(pipeline_path)  # Chỉ load trusted artifact do chính nhóm tạo.
    check_is_fitted(pipeline)
    assert np.array_equal(pipeline.classes_, np.arange(8))
    with np.load(baseline_file(prefix + 'golden_samples.npz'), allow_pickle=False) as golden:
        probe = pd.DataFrame(np.asarray(golden['X_raw_float32'], dtype=np.float64), columns=FEATURES)
        predicted = pipeline.predict_proba(probe)
        np.testing.assert_allclose(predicted, golden['probabilities'], rtol=1e-5, atol=1e-6)
        assert np.array_equal(predicted.argmax(axis=1), golden['probabilities'].argmax(axis=1))
    result = read_json(baseline_file('experiments/' + BASE_IDS[name] + '/result.json'))
    assert result['status'] == 'complete' and result['protocol_hash'] == BASE_PROTOCOL_HASH
    assert result['artifact_hashes']['model.joblib'] == bundle['model_sha256']
    BASE_PIPELINES[name] = pipeline
    BASE_SNAPSHOT[name] = {'experiment_id': BASE_IDS[name], 'config': config,
        'pipeline_sha256': bundle['pipeline_sha256'], 'model_sha256': bundle['model_sha256'],
        'original_fit_seconds': result['fit_seconds'], 'golden_check': True}
META_COLUMNS = [f'{model}__p_{label}' for model in BASE_ORDER for label in CLASSES]
print('Base golden checks: PASS. Features:', len(FEATURES), '; meta columns:', len(META_COLUMNS))


## 2. Khóa protocol và mở/tái sử dụng run

Pilot: DT, RF, XGBoost × `none/sqrt` × 3 folds. Tuning chỉ thuật toán thắng pilot,
tối đa 8 cấu hình. Chọn trong các candidate có precision ≥0,30 ở **cả BruteForce và Web-Based**;
macro-F1 trong 0,002 được coi gần hòa, ưu tiên min rare recall rồi fit time.
Không đạt floor thì dừng và xuất bảng để phân tích; không tự hạ floor.

Budget là giới hạn search đã checkpoint, không phải cam kết thời gian fit. Không ngắt một fit ở giữa;
kiểm tra budget trước mỗi fold. Chỉ chọn sau khi chạy đủ phase đã khóa.


In [ ]:
META_DEFAULTS = {
    'DT': {'max_depth': 6, 'min_samples_leaf': 10},
    'RF': {'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 5, 'max_features': 1.0},
    'XGBoost': {'max_depth': 2, 'min_child_weight': 5, 'n_estimators': 200,
        'learning_rate': 0.05, 'reg_lambda': 5.0, 'reg_alpha': 0.0,
        'subsample': 1.0, 'colsample_bytree': 1.0, 'max_bin': 256}}
TUNING_SPACES = {
    'DT': {'max_depth': [3, 6, 10], 'min_samples_leaf': [5, 10, 20, 50]},
    'RF': {'n_estimators': [100, 200], 'max_depth': [4, 8, 12],
           'min_samples_leaf': [3, 5, 10, 20], 'max_features': [0.5, 1.0]},
    'XGBoost': {'max_depth': [2, 3, 4], 'min_child_weight': [1, 5, 10],
        'learning_rate': [0.03, 0.05, 0.1], 'n_estimators': [100, 200, 400],
        'reg_lambda': [1.0, 5.0, 10.0], 'reg_alpha': [0.0, 0.1]}}
PROTOCOL = {'revision': NOTEBOOK_REVISION, 'author_branch': AUTHOR_BRANCH,
    'author_base_commit': AUTHOR_BASE_COMMIT, 'authoring_state': 'uncommitted notebook template',
    'template_source_sha256': TEMPLATE_SOURCE_SHA256,
    'source_run': SOURCE_RUN, 'pilot_run': PILOT_RUN, 'base_run': BASE_RUN,
    'source_config_sha256': INPUT_INTEGRITY['processing_config'],
    'pilot_manifest_sha256': INPUT_INTEGRITY['pilot_manifest'],
    'base_protocol_sha256': BASE_PROTOCOL_FILE_SHA256, 'base_protocol_hash': BASE_PROTOCOL_HASH,
    'base_models': BASE_SNAPSHOT, 'base_order': BASE_ORDER, 'classes': CLASSES,
    'raw_features': FEATURES, 'meta_columns': META_COLUMNS, 'base_mode': 'reuse frozen baseline',
    'cache_policy': 'require baseline cache; verify hashes and all row IDs; batch copy with checkpoint',
    'validation_mask_sha256': BASE_CHECKSUMS['val_float32_kept_processed_indices.npy'],
    'test_mask_sha256': BASE_CHECKSUMS['test_float32_kept_processed_indices.npy'],
    'versions': VERSIONS, 'n_threads': N_THREADS, 'batch_size': BATCH_SIZE,
    'seed': SEED, 'folds': N_FOLDS, 'group': 'BLAKE2b-128 raw 44 float32; canonical zero sign',
    'meta_defaults': META_DEFAULTS, 'tuning_spaces': TUNING_SPACES, 'tuning_trials': TUNING_TRIALS,
    'weights': ['none', 'sqrt'], 'precision_floor': MIN_RARE_PRECISION,
    'macro_f1_tolerance': MACRO_F1_TOLERANCE, 'decision': 'argmax; no tuning threshold',
    'run_seed_stability': RUN_SEED_STABILITY, 'stability_seeds': [2026, 3407],
    'benchmark_repeats': BENCHMARK_REPEATS,
    'history_disclosure': 'base selected with val; baseline val/test, recall and XAI viewed; final test historical',
    'preprocessing_fit_scope': 'full source train; base classifiers pilot 1M; meta masked val',
    'training_budget_caveat': 'meta learns additional val labels beyond base training data'}
PROTOCOL_HASH = object_hash(PROTOCOL)
WORKING_ROOT = Path('/kaggle/working/blending_experiments')
WORKING_ROOT.mkdir(parents=True, exist_ok=True)
if RESUME_FROM:
    resume = Path(RESUME_FROM)
    assert (resume / 'protocol.json').is_file(), 'RESUME_FROM phải trỏ tới thư mục run chứa protocol.json.'
    previous = read_json(resume / 'protocol.json')
    assert previous['protocol_hash'] == PROTOCOL_HASH and previous['protocol'] == PROTOCOL, (
        'Protocol/runtime/input thay đổi. Giữ config cũ để resume hoặc tạo run mới với RESUME_FROM=None.')
    assert resume.name == previous['run_id']
    OUT = WORKING_ROOT / previous['run_id']
    if resume.resolve() != OUT.resolve():
        if OUT.exists():
            assert read_json(OUT / 'protocol.json') == previous, 'Output đích khác run; không overwrite.'
            print('Đã có bản working của run; tiếp tục từ bản working, không chép đè input cũ.')
        else:
            shutil.copytree(resume, OUT)
    RUN_ID = previous['run_id']
else:
    RUN_ID = 'blending_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
    OUT = WORKING_ROOT / RUN_ID
    OUT.mkdir()
    atomic_json(OUT / 'protocol.json', {'run_id': RUN_ID, 'created_at': now(),
                'protocol_hash': PROTOCOL_HASH, 'protocol': PROTOCOL})
atomic_json(OUT / 'base_models_snapshot.json', BASE_SNAPSHOT)
atomic_json(OUT / 'meta_feature_schema.json', {'columns': META_COLUMNS, 'shape_columns': 24,
            'dtype': 'float32', 'base_order': BASE_ORDER, 'class_order': CLASSES})
atomic_json(OUT / 'hardware.json', {'platform': platform.platform(), 'cpu': platform.processor(),
    'logical_cpu': psutil.cpu_count(), 'physical_cpu': psutil.cpu_count(logical=False),
    'total_ram_mib': psutil.virtual_memory().total / 2**20, 'threads_used': N_THREADS,
    'device': 'cpu', 'versions': VERSIONS})
SEARCH_LOCKED = (OUT / 'frozen_blending.json').is_file()
print('Run:', OUT, '\nProtocol:', PROTOCOL_HASH, '\nFrozen:', SEARCH_LOCKED)


## 3. Xây Z_val: 24 cột và audit alignment toàn bộ dòng

Giữ nguyên mask baseline. Mọi model phải có cùng evaluation index, processed index,
source row ID và true label. Row IDs chỉ là traceability, không là feature/session ID.
Kiểm tra xác suất và đối chiếu pipeline trên mẫu kiểm định; không fit trên val ở bước này.
Cache Z checkpoint sau từng batch; batch chưa checkpoint sẽ được làm lại khi resume.


In [ ]:
def check_probabilities(p, rows=None):
    p = np.asarray(p)
    assert p.ndim == 2 and p.shape[1] == 8 and (rows is None or p.shape[0] == rows)
    assert np.isfinite(p).all() and p.min() >= -1e-7 and p.max() <= 1.000001
    np.testing.assert_allclose(p.sum(axis=1), 1.0, rtol=1e-5, atol=1e-5)


def prepare_split(split):
    assert split in ('val', 'test')
    if split == 'test':
        assert RUN_FINAL_TEST and (OUT / 'frozen_blending.json').is_file(), 'Test chỉ được mở sau freeze.'
        verify_frozen()
    audit = read_json(baseline_file(split + '_input_audit.json'))
    purge = read_json(baseline_file(split + '_float32_purge_audit.json'))
    assert purge['protocol_hash'] == BASE_PROTOCOL_HASH and purge['label_independent']
    assert purge['priority'] == ['train_union_of_all_budgets', 'validation', 'test']
    mask_path = baseline_file(split + '_float32_kept_processed_indices.npy')
    require_hash(mask_path, purge['kept_indices_sha256'])
    kept = np.load(mask_path, allow_pickle=False)
    assert kept.ndim == 1 and len(kept) == purge['after_rows'] and np.all(np.diff(kept) > 0)
    assert np.issubdtype(kept.dtype, np.integer)
    raw_path = PROCESSED_DIR / f'X_{split}_raw.npy'
    y_path = PROCESSED_DIR / f'y_{split}.npy'
    meta_path = PROCESSED_DIR / f'sample_metadata_{split}.pkl'
    INPUT_INTEGRITY[f'processing/{split}/raw'] = require_hash(raw_path, audit['raw_sha256'])
    INPUT_INTEGRITY[f'processing/{split}/target'] = require_hash(y_path, audit['target_sha256'])
    INPUT_INTEGRITY[f'processing/{split}/metadata'] = require_hash(meta_path, audit['metadata_sha256'])
    raw = np.load(raw_path, mmap_mode='r', allow_pickle=False)
    y_all = np.load(y_path, mmap_mode='r', allow_pickle=False)
    assert list(raw.shape) == audit['shape'] and raw.shape[1] == 44 and len(y_all) == len(raw)
    assert kept[0] >= 0 and kept[-1] < len(raw)
    y = np.asarray(y_all[kept], dtype=np.int64)
    assert np.array_equal(np.unique(y), np.arange(8))
    assert {c: int((y == i).sum()) for i, c in enumerate(CLASSES)} == purge['after_class_support']
    metadata = pd.read_pickle(meta_path)
    assert len(metadata) == len(raw)
    source_ids = metadata.iloc[kept]['source_row_index'].to_numpy(dtype=np.int64)
    assert np.array_equal(metadata.iloc[kept]['class_id'].to_numpy(dtype=np.int64), y)
    rows = pd.DataFrame({'evaluation_row_index': np.arange(len(y)), 'processed_row_index': kept,
                        'source_row_index': source_ids, 'source_split': split, 'true_class_id': y})
    reference_columns = list(rows.columns)
    paths, caches = {}, {}
    for name in BASE_ORDER:
        prefix = ('experiments/' + BASE_IDS[name] + '/validation/' if split == 'val'
                  else 'test/' + BASE_IDS[name] + '/')
        probs_path = baseline_file(prefix + 'probabilities.npy')
        preds_path = baseline_file(prefix + 'predictions.csv.gz')
        ids = pd.read_csv(preds_path)
        assert len(ids) == len(y)
        assert set(reference_columns + ['predicted_class_id']).issubset(ids.columns)
        for col in reference_columns:
            assert np.array_equal(ids[col].to_numpy(), rows[col].to_numpy()), f'{name}/{split}: lệch {col}'
        p = np.load(probs_path, mmap_mode='r', allow_pickle=False)
        assert p.shape == (len(y), 8) and p.dtype == np.float32
        for start in range(0, len(y), BATCH_SIZE):
            block = p[start:start + BATCH_SIZE]
            check_probabilities(block)
            assert np.array_equal(block.argmax(axis=1), ids['predicted_class_id'].to_numpy()[start:start + BATCH_SIZE])
        # Parity pipeline với cache trên mẫu đã khóa bằng seed, bao gồm đủ 8 lớp.
        rng = np.random.default_rng(SEED)
        probes = np.unique(np.concatenate([rng.choice(np.flatnonzero(y == i), min(8, (y == i).sum()), replace=False)
                                          for i in range(8)]))
        frame = pd.DataFrame(np.asarray(raw[kept[probes]], dtype=np.float64), columns=FEATURES)
        actual = BASE_PIPELINES[name].predict_proba(frame)
        np.testing.assert_allclose(actual, p[probes], rtol=1e-5, atol=1e-6)
        assert np.array_equal(actual.argmax(axis=1), p[probes].argmax(axis=1))
        paths[name] = probs_path
        caches[name] = p
        del ids
    folder = OUT / ('meta_data' if split == 'val' else 'test_data')
    folder.mkdir(parents=True, exist_ok=True)
    identity = {'protocol_hash': PROTOCOL_HASH, 'split': split,
        'mask_sha256': sha256(mask_path), 'cache_sha256': {n: sha256(p) for n, p in paths.items()},
        'raw_sha256': audit['raw_sha256'], 'y_sha256': audit['target_sha256'], 'rows': len(y), 'columns': META_COLUMNS}
    z_path = folder / ('Z_val.npy' if split == 'val' else 'Z_test.npy')
    if (folder / 'complete.json').exists():
        record = verify_seal(folder)
        assert record['identity'] == identity
        z = np.load(z_path, mmap_mode='r', allow_pickle=False)
    else:
        progress_path = folder / 'z_progress.json'
        if progress_path.exists():
            progress = read_json(progress_path)
            assert progress['identity'] == identity
            z = np.lib.format.open_memmap(z_path, mode='r+')
            start_at = progress['next_row']
        else:
            z = np.lib.format.open_memmap(z_path, mode='w+', dtype=np.float32, shape=(len(y), 24))
            start_at = 0
            atomic_json(progress_path, {'identity': identity, 'next_row': 0})
        assert z.shape == (len(y), 24) and z.dtype == np.float32
        for start in range(start_at, len(y), BATCH_SIZE):
            stop = min(start + BATCH_SIZE, len(y))
            z[start:stop] = np.concatenate([caches[n][start:stop] for n in BASE_ORDER], axis=1)
            z.flush()
            atomic_json(progress_path, {'identity': identity, 'next_row': stop})
        atomic_npy(folder / 'y_meta.npy', y)
        rows.to_csv(folder / 'meta_row_ids.csv', index=False)
        atomic_json(folder / 'probability_alignment_audit.json', {**identity, 'all_row_ids_checked': True,
            'probabilities_checked_all_rows': True, 'pipeline_cache_sample_parity': True, 'support': purge['after_class_support']})
        progress_path.unlink()
        del z
        seal(folder, extra={'identity': identity})
        z = np.load(z_path, mmap_mode='r', allow_pickle=False)
    assert z.shape == (len(y), 24) and z.dtype == np.float32
    # Verify toàn bộ Z kể cả các batch đã checkpoint trước khi kernel bị dừng.
    for start in range(0, len(y), BATCH_SIZE):
        expected_block = np.concatenate([caches[n][start:start + BATCH_SIZE] for n in BASE_ORDER], axis=1)
        assert np.array_equal(z[start:start + BATCH_SIZE], expected_block), f'{split}: Z/cache lệch tại row {start}'
    assert np.array_equal(np.load(folder / 'y_meta.npy', allow_pickle=False), y)
    atomic_json(OUT / 'input_integrity.json', INPUT_INTEGRITY)
    del metadata, caches
    gc.collect()
    return raw, kept, y, rows, z


XVAL_RAW, VAL_KEPT, Y_META, META_ROWS, ZVAL = prepare_split('val')
print('Z_val:', ZVAL.shape, ZVAL.dtype, '; MiB:', ZVAL.nbytes / 2**20)


## 4. Group CV: raw-float32 fingerprints, 3 folds

Bản lặp raw vector cùng group nằm trong một held-out fold. Đây chưa phải group theo device/session/time.
Dừng nếu bất kỳ fold thiếu lớp; lưu support và số group hiếm thực tế.
Audit thêm Z trùng nhưng khác nhãn để mô tả giới hạn biểu diễn; không xóa mẫu vì audit này.


In [ ]:
FOLD_DIR = OUT / 'folds'
FOLD_DIR.mkdir(exist_ok=True)
if (FOLD_DIR / 'complete.json').exists():
    verify_seal(FOLD_DIR)
    GROUPS = np.load(FOLD_DIR / 'groups.npy', allow_pickle=False)
    FOLD_ID = np.load(FOLD_DIR / 'fold_id.npy', allow_pickle=False)
else:
    fingerprints = np.empty(len(Y_META), dtype='V16')
    for start in range(0, len(Y_META), BATCH_SIZE):
        block = np.array(XVAL_RAW[VAL_KEPT[start:start + BATCH_SIZE]], dtype='<f4', order='C')
        assert np.isfinite(block).all(), 'Raw arrays đã xử lý phải finite trước hashing.'
        block[block == 0] = 0.0
        fingerprints[start:start + len(block)] = [hashlib.blake2b(row.tobytes(), digest_size=16).digest() for row in block]
    _, GROUPS = np.unique(fingerprints, return_inverse=True)
    del fingerprints
    FOLD_ID = np.full(len(Y_META), -1, dtype=np.int8)
    splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    supports = []
    for fold, (train_idx, hold_idx) in enumerate(splitter.split(np.zeros(len(Y_META)), Y_META, GROUPS)):
        assert np.array_equal(np.unique(Y_META[train_idx]), np.arange(8)), 'Meta train fold thiếu lớp.'
        assert np.array_equal(np.unique(Y_META[hold_idx]), np.arange(8)), 'Held-out fold thiếu lớp.'
        assert np.intersect1d(GROUPS[train_idx], GROUPS[hold_idx]).size == 0
        assert np.all(FOLD_ID[hold_idx] == -1)
        FOLD_ID[hold_idx] = fold
        supports.append({'fold': fold, 'train_rows': len(train_idx), 'heldout_rows': len(hold_idx),
            'heldout_support': {c: int((Y_META[hold_idx] == i).sum()) for i, c in enumerate(CLASSES)},
            'heldout_groups_by_class': {c: int(len(np.unique(GROUPS[hold_idx][Y_META[hold_idx] == i])))
                                       for i, c in enumerate(CLASSES)}})
    assert np.all(FOLD_ID >= 0)
    atomic_npy(FOLD_DIR / 'groups.npy', GROUPS)
    atomic_npy(FOLD_DIR / 'fold_id.npy', FOLD_ID)
    # Các vector Z giống nhau có thể chứa nhãn khác nhau, không lọc chúng.
    z_bytes = np.ascontiguousarray(ZVAL).view(np.dtype((np.void, 24 * 4))).ravel()
    _, z_group = np.unique(z_bytes, return_inverse=True)
    low = np.full(int(z_group.max()) + 1, 8, dtype=np.int8)
    high = np.full(len(low), -1, dtype=np.int8)
    np.minimum.at(low, z_group, Y_META)
    np.maximum.at(high, z_group, Y_META)
    mixed = low != high
    atomic_json(FOLD_DIR / 'representation_audit.json', {
        'z_unique_vectors': len(low), 'z_vectors_with_multiple_labels': int(mixed.sum()),
        'rows_in_conflicting_z_vectors': int(mixed[z_group].sum()), 'policy': 'report only; no filtering'})
    atomic_json(FOLD_DIR / 'meta_folds.json', {'protocol_hash': PROTOCOL_HASH, 'folds': supports,
        'group_count': int(GROUPS.max()) + 1, 'group_definition': PROTOCOL['group'],
        'limitation': 'No proven session/device/time independence'})
    del z_bytes, z_group, low, high, mixed
    seal(FOLD_DIR)
assert len(FOLD_ID) == len(Y_META) and len(GROUPS) == len(Y_META)
for fold in range(N_FOLDS):
    assert np.array_equal(np.unique(Y_META[FOLD_ID == fold]), np.arange(8))
# Verify group allocation again on resume.
minimum = np.full(int(GROUPS.max()) + 1, N_FOLDS, dtype=np.int8)
maximum = np.full(len(minimum), -1, dtype=np.int8)
np.minimum.at(minimum, GROUPS, FOLD_ID)
np.maximum.at(maximum, GROUPS, FOLD_ID)
assert np.array_equal(minimum, maximum), 'Group bị chia sang nhiều folds.'
del minimum, maximum
FOLD_HASH = sha256(FOLD_DIR / 'fold_id.npy')
display(pd.DataFrame(read_json(FOLD_DIR / 'meta_folds.json')['folds']))
print('Z representation audit:', read_json(FOLD_DIR / 'representation_audit.json'))


## 5. Metrics, checkpoint và runner CV

Mỗi fold lưu model, probabilities, labels/weights/timing và completion marker có checksums.
Chỉ khi đủ 3 folds mới tổng hợp OOF và cho phép candidate tham gia lựa chọn.
Budget được ghi sau mỗi fold; hard kill có thể mất accounting của fold chưa checkpoint.
Thời gian search đã lưu là giới hạn mềm; final fit/export/test có timing riêng.


In [ ]:
class PeakRSS:
    def __enter__(self):
        self.process = psutil.Process()
        self.baseline = self.process.memory_info().rss / 2**20
        self.peak = self.baseline
        self.stop = threading.Event()
        def sample():
            while not self.stop.wait(0.05):
                self.peak = max(self.peak, self.process.memory_info().rss / 2**20)
        self.worker = threading.Thread(target=sample, daemon=True)
        self.worker.start()
        return self

    def __exit__(self, *args):
        self.peak = max(self.peak, self.process.memory_info().rss / 2**20)
        self.stop.set()
        self.worker.join()


def metrics(y, probabilities):
    check_probabilities(probabilities, len(y))
    pred = np.asarray(probabilities).argmax(axis=1)
    report = classification_report(y, pred, labels=np.arange(8), target_names=CLASSES,
                                   output_dict=True, zero_division=0)
    cm = confusion_matrix(y, pred, labels=np.arange(8))
    result = {'rows': len(y), 'accuracy': float((pred == y).mean()),
        'macro_precision': report['macro avg']['precision'], 'macro_recall': report['macro avg']['recall'],
        'macro_f1': report['macro avg']['f1-score'], 'weighted_f1': report['weighted avg']['f1-score'],
        'log_loss': float(log_loss(y, probabilities, labels=np.arange(8))),
        'normal_false_alarm_rate': float(cm[0, 1:].sum() / max(1, cm[0].sum()))}
    for label in ['BruteForce', 'Web-Based']:
        for key in ['precision', 'recall', 'f1-score']:
            result[key.replace('-score', '') + '_' + label] = report[label][key]
    result['min_rare_recall'] = min(result['recall_BruteForce'], result['recall_Web-Based'])
    return result, report, cm


def export_metrics(folder, y, p, title):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    result, report, cm = metrics(y, p)
    atomic_json(folder / 'metrics.json', result)
    atomic_json(folder / 'classification_report.json', report)
    pd.DataFrame([{'class': c, **report[c]} for c in CLASSES]).to_csv(folder / 'per_class_metrics.csv', index=False)
    pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(folder / 'confusion_matrix_counts.csv')
    normalized = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    pd.DataFrame(normalized, index=CLASSES, columns=CLASSES).to_csv(folder / 'confusion_matrix_normalized.csv')
    fig, ax = plt.subplots(figsize=(10, 8))
    plot = ax.imshow(normalized, vmin=0, vmax=1, cmap='Blues')
    ax.set(xticks=np.arange(8), yticks=np.arange(8), xticklabels=CLASSES, yticklabels=CLASSES,
           xlabel='Predicted', ylabel='True', title=title)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    for i in range(8):
        for j in range(8):
            ax.text(j, i, f'{cm[i,j]}\n{normalized[i,j]:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if normalized[i,j] > 0.5 else 'black')
    fig.colorbar(plot, ax=ax)
    fig.tight_layout()
    fig.savefig(folder / 'confusion_matrix.png', dpi=150)
    plt.close(fig)
    return result


def make_meta(spec):
    params = {**spec['params'], 'random_state': spec['seed']}
    if spec['algorithm'] == 'DT':
        return DecisionTreeClassifier(**params)
    if spec['algorithm'] == 'RF':
        return RandomForestClassifier(**params, n_jobs=N_THREADS)
    return XGBClassifier(**params, objective='multi:softprob', num_class=8, tree_method='hist',
                         device='cpu', eval_metric='mlogloss', n_jobs=N_THREADS)


def sample_weights(y, mode):
    counts = np.bincount(y, minlength=8).astype(np.float64)
    assert np.all(counts > 0)
    class_weights = np.ones(8) if mode == 'none' else np.sqrt(len(y) / (8 * counts))
    assert mode in ('none', 'sqrt')
    class_weights /= class_weights[y].mean()
    return class_weights[y].astype(np.float32), {c: float(class_weights[i]) for i, c in enumerate(CLASSES)}


def make_spec(algorithm, weight='none', params=None, seed=SEED):
    return {'algorithm': algorithm, 'weight': weight, 'params': params or META_DEFAULTS[algorithm], 'seed': seed}


LEDGER_PATH = OUT / 'search_budget.json'
LEDGER = read_json(LEDGER_PATH) if LEDGER_PATH.exists() else {'events': [], 'checkpointed_seconds': 0.0}


class SearchBudgetStop(RuntimeError):
    pass


def before_search_fit():
    if LEDGER['checkpointed_seconds'] >= SEARCH_BUDGET_HOURS * 3600:
        raise SearchBudgetStop('Đã hết budget search. Save Version giữ output, resume run này và tăng SEARCH_BUDGET_HOURS. '
                               'Không chọn candidate từ phase còn thiếu fold/trial.')


def account_search(identity, seconds):
    LEDGER['events'].append({'identity': identity, 'seconds': float(seconds), 'at': now()})
    LEDGER['checkpointed_seconds'] += float(seconds)
    atomic_json(LEDGER_PATH, LEDGER)


RESULTS = {}
for marker in (OUT / 'trials').glob('*/complete.json'):
    verify_seal(marker.parent)
    result = read_json(marker.parent / 'result.json')
    assert result['protocol_hash'] == PROTOCOL_HASH and result['fold_hash'] == FOLD_HASH
    RESULTS[result['trial_id']] = result


def result_table():
    rows = [{'trial_id': r['trial_id'], 'algorithm': r['spec']['algorithm'], 'weight': r['spec']['weight'],
             'seed': r['spec']['seed'], 'fit_seconds': r['fit_seconds'],
             'predict_seconds': r['predict_seconds'], **r['metrics']} for r in RESULTS.values()]
    table = pd.DataFrame(rows)
    if len(table):
        table['feasible'] = ((table['precision_BruteForce'] >= MIN_RARE_PRECISION) &
                             (table['precision_Web-Based'] >= MIN_RARE_PRECISION))
        # Pareto among macro-F1 / both rare recalls; precision feasibility shown separately.
        objectives = table[['macro_f1', 'recall_BruteForce', 'recall_Web-Based']].to_numpy()
        table['pareto'] = [not np.any(np.all(objectives >= point, axis=1) & np.any(objectives > point, axis=1))
                           for point in objectives]
        table = table.sort_values(['macro_f1', 'min_rare_recall'], ascending=False)
    table.to_csv(OUT / 'meta_cv_results.csv', index=False)
    return table


def run_cv(spec):
    trial_id = 'meta_' + spec['algorithm'] + '_' + object_hash({'spec': spec, 'protocol': PROTOCOL_HASH})[:16]
    if trial_id in RESULTS:
        return RESULTS[trial_id]
    assert not SEARCH_LOCKED, 'Đã freeze: không chạy search mới.'
    folder = OUT / 'trials' / trial_id
    folder.mkdir(parents=True, exist_ok=True)
    atomic_json(folder / 'spec.json', spec)
    oof = np.full((len(Y_META), 8), np.nan, dtype=np.float32)
    fold_results = []
    for fold in range(N_FOLDS):
        train_idx = np.flatnonzero(FOLD_ID != fold)
        held_idx = np.flatnonzero(FOLD_ID == fold)
        fold_dir = folder / f'fold_{fold}'
        fold_dir.mkdir(exist_ok=True)
        identity = {'protocol_hash': PROTOCOL_HASH, 'fold_hash': FOLD_HASH, 'spec': spec, 'fold': fold}
        if (fold_dir / 'complete.json').exists():
            record = verify_seal(fold_dir)
            assert record['identity'] == identity
            p = np.load(fold_dir / 'probabilities.npy', allow_pickle=False)
            info = read_json(fold_dir / 'result.json')
        else:
            before_search_fit()
            started = time.perf_counter()
            try:
                x_train = np.asarray(ZVAL[train_idx], dtype=np.float32)
                x_hold = np.asarray(ZVAL[held_idx], dtype=np.float32)
                weights, class_weights = sample_weights(Y_META[train_idx], spec['weight'])
                model = make_meta(spec)
                with threadpool_limits(limits=N_THREADS), PeakRSS() as memory:
                    fit_start = time.perf_counter()
                    model.fit(x_train, Y_META[train_idx], sample_weight=weights)
                    fit_seconds = time.perf_counter() - fit_start
                    assert np.array_equal(model.classes_, np.arange(8))
                    predict_start = time.perf_counter()
                    p = model.predict_proba(x_hold).astype(np.float32)
                    predict_seconds = time.perf_counter() - predict_start
                check_probabilities(p, len(held_idx))
                info = {'fold': fold, 'fit_seconds': fit_seconds, 'predict_seconds': predict_seconds,
                    'baseline_rss_mib': memory.baseline, 'peak_rss_mib': memory.peak,
                    'rss_increase_mib': memory.peak - memory.baseline,
                    'class_weights': class_weights, 'train_rows': len(train_idx), 'heldout_rows': len(held_idx),
                    'metrics': metrics(Y_META[held_idx], p)[0]}
                atomic_npy(fold_dir / 'heldout_indices.npy', held_idx)
                atomic_npy(fold_dir / 'probabilities.npy', p)
                atomic_joblib(fold_dir / 'model.joblib', model)
                atomic_json(fold_dir / 'result.json', info)
                seal(fold_dir, extra={'identity': identity})
                del model, x_train, x_hold, weights
                gc.collect()
            finally:
                account_search(trial_id + f'/fold_{fold}', time.perf_counter() - started)
        check_probabilities(p, len(held_idx))
        assert np.all(np.isnan(oof[held_idx]))
        oof[held_idx] = p
        fold_results.append(info)
        print(trial_id, 'fold', fold + 1, '/', N_FOLDS, 'macro-F1:', round(info['metrics']['macro_f1'], 5), flush=True)
    check_probabilities(oof, len(Y_META))
    atomic_npy(folder / 'meta_oof_probabilities.npy', oof)
    atomic_npy(folder / 'predicted_labels.npy', oof.argmax(axis=1))
    stats = export_metrics(folder / 'oof', Y_META, oof, trial_id + ' — development OOF')
    result = {'trial_id': trial_id, 'spec': spec, 'protocol_hash': PROTOCOL_HASH, 'fold_hash': FOLD_HASH,
        'metrics': stats, 'fit_seconds': sum(r['fit_seconds'] for r in fold_results),
        'predict_seconds': sum(r['predict_seconds'] for r in fold_results),
        'max_peak_rss_mib': max(r['peak_rss_mib'] for r in fold_results), 'fold_results': fold_results,
        'fold_macro_f1_mean': float(np.mean([r['metrics']['macro_f1'] for r in fold_results])),
        'fold_macro_f1_std': float(np.std([r['metrics']['macro_f1'] for r in fold_results], ddof=1)),
        'interpretation': 'development OOF used in selection; not independent confirmation'}
    atomic_json(folder / 'result.json', result)
    seal(folder, extra={'protocol_hash': PROTOCOL_HASH})
    RESULTS[trial_id] = result
    result_table()
    del oof
    gc.collect()
    return result


def select_candidate(candidates):
    feasible = [r for r in candidates if r['metrics']['precision_BruteForce'] >= MIN_RARE_PRECISION
                and r['metrics']['precision_Web-Based'] >= MIN_RARE_PRECISION]
    if not feasible:
        display(result_table())
        atomic_json(OUT / 'selection_failure.json', {'at': now(), 'reason': 'No candidate meets both rare precision floors',
                    'precision_floor': MIN_RARE_PRECISION, 'protocol_hash': PROTOCOL_HASH})
        raise RuntimeError('Không candidate đạt precision floor của cả hai lớp hiếm. Đọc meta_cv_results.csv; '
                           'chưa fit/freeze/test. Nếu đổi protocol, tạo run mới.')
    best_macro = max(r['metrics']['macro_f1'] for r in feasible)
    close = [r for r in feasible if r['metrics']['macro_f1'] >= best_macro - MACRO_F1_TOLERANCE]
    return sorted(close, key=lambda r: (-r['metrics']['min_rare_recall'], r['fit_seconds'],
                                        r['predict_seconds'], r['trial_id']))[0]


## 6. Đối chứng và error coverage của ba base-model

Cả 3 base và equal soft voting dùng cùng val đã purge. Điểm base/soft vote là development;
OOF meta được ghép từ held-out folds. Báo lỗi đồng thời để biết tầng meta có tín hiệu sửa lỗi hay không.


In [ ]:
CONTROLS_DIR = OUT / 'development_controls'
if (CONTROLS_DIR / 'complete.json').exists():
    verify_seal(CONTROLS_DIR)
    CONTROLS = read_json(CONTROLS_DIR / 'comparison.json')
else:
    CONTROLS_DIR.mkdir(exist_ok=True)
    CONTROLS = []
    base_predictions = np.column_stack([ZVAL[:, i * 8:(i + 1) * 8].argmax(axis=1) for i in range(3)])
    for i, name in enumerate(BASE_ORDER):
        stats = export_metrics(CONTROLS_DIR / name, Y_META, ZVAL[:, i*8:(i+1)*8], name + ' — development val')
        CONTROLS.append({'model': name, 'kind': 'base validation (historically selected)', **stats})
    soft = np.mean(np.asarray(ZVAL).reshape(-1, 3, 8), axis=1, dtype=np.float32)
    stats = export_metrics(CONTROLS_DIR / 'SoftVoting', Y_META, soft, 'Equal soft voting — development val')
    CONTROLS.append({'model': 'SoftVoting', 'kind': 'fixed equal averaging', **stats})
    coverage = []
    for i, label in enumerate(CLASSES):
        correct = base_predictions[Y_META == i] == i
        coverage.append({'class': label, 'support': int((Y_META == i).sum()),
            'all_base_wrong': int((~correct.any(axis=1)).sum()), 'any_base_correct': int(correct.any(axis=1).sum()),
            'all_base_correct': int(correct.all(axis=1).sum())})
    pd.DataFrame(coverage).to_csv(CONTROLS_DIR / 'base_error_coverage.csv', index=False)
    atomic_json(CONTROLS_DIR / 'comparison.json', CONTROLS)
    seal(CONTROLS_DIR)
    del soft, base_predictions
pd.DataFrame(CONTROLS).to_csv(OUT / 'development_controls.csv', index=False)
display(pd.DataFrame(CONTROLS)[['model', 'macro_f1', 'precision_BruteForce', 'recall_BruteForce',
                              'precision_Web-Based', 'recall_Web-Based', 'accuracy']])


## 7. Pilot meta: 18 fits có checkpoint

Có thể mất thời gian dù chỉ 24 cột vì mỗi fold chứa hơn nửa triệu dòng.
Dòng log xuất sau mỗi fold; Save Version giữ output để resume, không chạy lại các fold đã hoàn tất.


In [ ]:
if not SEARCH_LOCKED:
    pilot = [run_cv(make_spec(algorithm, weight)) for algorithm in BASE_ORDER for weight in ['none', 'sqrt']]
    pilot_winner = select_candidate(pilot)
    pilot_state = {'protocol_hash': PROTOCOL_HASH, 'trial_ids': [r['trial_id'] for r in pilot],
                   'selected_trial': pilot_winner['trial_id'], 'selected_spec': pilot_winner['spec']}
    if (OUT / 'pilot_selected.json').exists():
        assert read_json(OUT / 'pilot_selected.json') == pilot_state
    atomic_json(OUT / 'pilot_selected.json', pilot_state)
else:
    pilot_state = read_json(OUT / 'pilot_selected.json')
    pilot_winner = RESULTS[pilot_state['selected_trial']]
print('Pilot winner:', pilot_winner['spec'])
display(result_table())


## 8. Tuning có giới hạn và chọn candidate cuối

Chỉ tuning thuật toán/weight thắng pilot; seed và folds giữ cố định.
Tối đa 8 cấu hình sampled × 3 folds. Không hạ precision floor và không gọi test ở đây.


In [ ]:
if not SEARCH_LOCKED:
    algorithm = pilot_winner['spec']['algorithm']
    tuning_specs = [make_spec(algorithm, pilot_winner['spec']['weight'],
        {**pilot_winner['spec']['params'], **params})
        for params in ParameterSampler(TUNING_SPACES[algorithm], n_iter=TUNING_TRIALS, random_state=SEED)]
    tuning = [run_cv(spec) for spec in tuning_specs]
    selection_pool = [RESULTS[t] for t in pilot_state['trial_ids']] + tuning
    SELECTED = select_candidate(selection_pool)
    selected_state = {'protocol_hash': PROTOCOL_HASH, 'fold_hash': FOLD_HASH,
        'pilot_trial_ids': pilot_state['trial_ids'], 'tuning_trial_ids': [r['trial_id'] for r in tuning],
        'selected_trial': SELECTED['trial_id'], 'spec': SELECTED['spec'],
        'metrics': SELECTED['metrics'], 'selection_rule': 'precision floor -> macro tolerance -> min rare recall -> fit/predict time'}
    if (OUT / 'selected_meta.json').exists():
        assert read_json(OUT / 'selected_meta.json') == selected_state
    atomic_json(OUT / 'selected_meta.json', selected_state)
else:
    selected_state = read_json(OUT / 'selected_meta.json')
    SELECTED = RESULTS[selected_state['selected_trial']]
print('Final meta candidate:', SELECTED['spec'])
display(result_table())


## 9. Seed stability (tùy chọn) và chẩn đoán development

Seeds 2026/3407 dùng cùng config/folds; seed cuối vẫn là 42, không chọn seed theo test.
Khoảng tin cậy recall dưới đây bootstrap **group raw vector trong từng lớp hiếm**,
không coi 3 seeds/model là các mẫu độc lập; vẫn là development đã dùng lựa chọn.


In [ ]:
if RUN_SEED_STABILITY:
    stability = [SELECTED]
    for other_seed in [2026, 3407]:
        spec = {**SELECTED['spec'], 'seed': other_seed}
        if SEARCH_LOCKED:
            trial_id = 'meta_' + spec['algorithm'] + '_' + object_hash({'spec': spec, 'protocol': PROTOCOL_HASH})[:16]
            stability.append(RESULTS[trial_id])
        else:
            stability.append(run_cv(spec))
    pd.DataFrame([{'seed': r['spec']['seed'], **r['metrics']} for r in stability]).to_csv(OUT / 'seed_stability.csv', index=False)
    display(pd.read_csv(OUT / 'seed_stability.csv'))


def rare_recall_ci(y, pred, groups, repeats=500):
    rng = np.random.default_rng(SEED)
    records = []
    for i in [1, 7]:
        idx = np.flatnonzero(y == i)
        _, inverse = np.unique(groups[idx], return_inverse=True)
        counts = np.bincount(inverse)
        successes = np.bincount(inverse, weights=(pred[idx] == i).astype(np.float64))
        draws = rng.integers(len(counts), size=(repeats, len(counts)))
        rates = successes[draws].sum(axis=1) / counts[draws].sum(axis=1)
        records.append({'class': CLASSES[i], 'recall': float((pred[idx] == i).mean()),
            'ci_lower_95': float(np.quantile(rates, 0.025)), 'ci_upper_95': float(np.quantile(rates, 0.975)),
            'raw_vector_groups': len(counts), 'support': len(idx), 'bootstrap_repeats': repeats,
            'limitation': 'raw-vector group bootstrap; not session/device independent'})
    return records


SELECTED_OOF = np.load(OUT / 'trials' / SELECTED['trial_id'] / 'meta_oof_probabilities.npy', mmap_mode='r')
BLEND_OOF_PRED = SELECTED_OOF.argmax(axis=1)
pd.DataFrame(rare_recall_ci(Y_META, BLEND_OOF_PRED, GROUPS)).to_csv(OUT / 'development_rare_recall_ci.csv', index=False)
comparison = pd.DataFrame(CONTROLS + [{'model': 'Blending', 'kind': 'selected meta development OOF', **SELECTED['metrics']}])
comparison.to_csv(OUT / 'development_comparison.csv', index=False)
errors = META_ROWS.copy()
errors['fold'] = FOLD_ID
for i, name in enumerate(BASE_ORDER):
    errors[name + '_prediction'] = ZVAL[:, i*8:(i+1)*8].argmax(axis=1)
errors['blending_oof_prediction'] = BLEND_OOF_PRED
errors.loc[errors['true_class_id'].isin([1, 7])].to_csv(OUT / 'development_rare_cases.csv', index=False)
display(comparison[['model', 'macro_f1', 'precision_BruteForce', 'recall_BruteForce',
                    'precision_Web-Based', 'recall_Web-Based', 'accuracy']])
print('Nếu blending thua đối chứng hoặc chỉ đánh đổi precision: báo kết quả âm; không chọn lại theo test.')


## 10. Fit meta cuối trên toàn Z_val

Giữ nguyên ba base-model; chỉ meta-model học thêm val. Lưu fit time và peak RSS riêng.
Nếu resume một final fit hoàn tất, kiểm tra checksum rồi nạp lại, không fit lần hai.


In [ ]:
FINAL_DIR = OUT / 'final_meta'
FINAL_DIR.mkdir(exist_ok=True)
FINAL_IDENTITY = {'protocol_hash': PROTOCOL_HASH, 'selected_trial': SELECTED['trial_id'],
                  'spec': SELECTED['spec'], 'training_rows': len(Y_META), 'z_sha256': sha256(OUT / 'meta_data/Z_val.npy')}
if (FINAL_DIR / 'complete.json').exists():
    record = verify_seal(FINAL_DIR)
    assert record['identity'] == FINAL_IDENTITY
    META_MODEL = joblib.load(FINAL_DIR / 'meta_model.joblib')
else:
    assert not SEARCH_LOCKED
    META_MODEL = make_meta(SELECTED['spec'])
    weights, class_weights = sample_weights(Y_META, SELECTED['spec']['weight'])
    with threadpool_limits(limits=N_THREADS), PeakRSS() as memory:
        started = time.perf_counter()
        META_MODEL.fit(np.asarray(ZVAL), Y_META, sample_weight=weights)
        final_fit_seconds = time.perf_counter() - started
    assert np.array_equal(META_MODEL.classes_, np.arange(8))
    atomic_joblib(FINAL_DIR / 'meta_model.joblib', META_MODEL)
    atomic_json(FINAL_DIR / 'fit_summary.json', {'spec': SELECTED['spec'], 'fit_seconds': final_fit_seconds,
        'training_rows': len(Y_META), 'class_weights': class_weights,
        'support': {c: int((Y_META == i).sum()) for i, c in enumerate(CLASSES)},
        'baseline_rss_mib': memory.baseline, 'peak_rss_mib': memory.peak,
        'rss_increase_mib': memory.peak - memory.baseline,
        'base_training_rows_per_model': 1000000,
        'fit_scope': 'meta all masked val; base pilot 1M; preprocessing full source train'})
    # Save/load parity của meta trên Z, trước khi export pipeline raw.
    loaded_meta = joblib.load(FINAL_DIR / 'meta_model.joblib')
    golden_indices = np.unique(np.linspace(0, len(Y_META) - 1, 64, dtype=np.int64))
    np.testing.assert_allclose(loaded_meta.predict_proba(ZVAL[golden_indices]),
                               META_MODEL.predict_proba(ZVAL[golden_indices]), rtol=1e-6, atol=1e-7)
    del loaded_meta, weights
    seal(FINAL_DIR, extra={'identity': FINAL_IDENTITY})
print('Final meta:', read_json(FINAL_DIR / 'fit_summary.json'))


## 11. Export module suy luận ổn định và benchmark toàn pipeline

Wrapper có `fit`/fitted-state/classes theo giao thức sklearn; `.fit` của wrapper bị khóa để tránh
vô tình fit lại base/meta trên query. Class được định nghĩa trong `blending_module.py`, không ở `__main__`.
Khi gọi raw API, chạy preprocessing cố định của **từng base** rồi ghép 24 score và gọi meta.

Handoff gồm module, pipeline serialize, bản sao base/meta, schema, train-median config, requirements,
golden và CSV minh họa. Kiên chỉ cần bundle + query cho demo; không cần tải toàn dataset.


In [ ]:
RUNTIME_MODULE = '"""Frozen 8-class holdout blending; load only trusted bundles created by this project."""\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import BaseEstimator, ClassifierMixin\nfrom sklearn.utils.validation import check_is_fitted\n\nBASE_ORDER = (\'DT\', \'RF\', \'XGBoost\')\n\n\ndef _sha(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\nclass BlendingClassifier(ClassifierMixin, BaseEstimator):\n    def __init__(self, base_pipelines=None, meta_estimator=None, features=None, class_names=None):\n        self.base_pipelines = base_pipelines\n        self.meta_estimator = meta_estimator\n        self.features = features\n        self.class_names = class_names\n\n    def initialize_pretrained(self):\n        if self.base_pipelines is None or self.meta_estimator is None:\n            raise ValueError(\'Require fitted base pipelines and meta estimator.\')\n        if list(self.base_pipelines) != list(BASE_ORDER):\n            raise ValueError(\'Base order must be DT, RF, XGBoost.\')\n        for model in list(self.base_pipelines.values()) + [self.meta_estimator]:\n            check_is_fitted(model)\n            if not np.array_equal(model.classes_, np.arange(8)):\n                raise ValueError(\'Class order must be IDs 0..7.\')\n        if len(self.features) != 44 or len(self.class_names) != 8:\n            raise ValueError(\'Require 44 raw features and 8 classes.\')\n        self.classes_ = np.arange(8)\n        self.n_features_in_ = len(self.features)\n        self.feature_names_in_ = np.asarray(self.features, dtype=object)\n        self.is_fitted_ = True\n        return self\n\n    def fit(self, X, y=None, **kwargs):\n        raise RuntimeError(\'This wrapper is inference-only. Fit bases on train and meta on val in the Kaggle notebook.\')\n\n    def __sklearn_is_fitted__(self):\n        return bool(getattr(self, \'is_fitted_\', False))\n\n    def _frame(self, frame):\n        check_is_fitted(self)\n        if isinstance(frame, pd.DataFrame):\n            if frame.columns.duplicated().any():\n                raise ValueError(\'Duplicate column names.\')\n            missing = [name for name in self.features if name not in frame.columns]\n            if missing:\n                raise ValueError(f\'Missing features: {missing}\')\n            selected = frame.loc[:, self.features].apply(pd.to_numeric, errors=\'raise\')\n        else:\n            array = np.asarray(frame)\n            if array.ndim != 2 or array.shape[1] != len(self.features):\n                raise ValueError(\'Array must have shape (n,44) in recorded feature order.\')\n            selected = pd.DataFrame(array, columns=self.features).apply(pd.to_numeric, errors=\'raise\')\n        # Reuse per-base fixed train-median preprocessors; no fit on query.\n        selected = selected.astype(np.float64).replace([np.inf, -np.inf], np.nan)\n        return selected\n\n    def build_meta_features(self, frame):\n        selected = self._frame(frame)\n        if len(selected) == 0:\n            return np.empty((0, 24), dtype=np.float32)\n        blocks = []\n        for name in BASE_ORDER:\n            p = np.asarray(self.base_pipelines[name].predict_proba(selected), dtype=np.float32)\n            if p.shape != (len(selected), 8) or not np.isfinite(p).all():\n                raise ValueError(f\'Invalid probabilities from {name}.\')\n            if not np.allclose(p.sum(axis=1), 1.0, rtol=1e-5, atol=1e-5):\n                raise ValueError(f\'Invalid probability sums from {name}.\')\n            blocks.append(p)\n        return np.ascontiguousarray(np.concatenate(blocks, axis=1), dtype=np.float32)\n\n    def predict_proba(self, X):\n        z = self.build_meta_features(X)\n        if len(z) == 0:\n            return np.empty((0, 8), dtype=np.float32)\n        p = np.asarray(self.meta_estimator.predict_proba(z), dtype=np.float32)\n        if p.shape != (len(z), 8) or not np.isfinite(p).all():\n            raise ValueError(\'Invalid meta probabilities.\')\n        return p\n\n    def predict(self, X):\n        return self.classes_[self.predict_proba(X).argmax(axis=1)]\n\n    def predict_proba_raw(self, frame):\n        return self.predict_proba(frame)\n\n    def predict_raw(self, frame):\n        return self.predict(frame)\n\n\ndef load_bundle(bundle_dir):\n    root = Path(bundle_dir)\n    manifest = json.loads((root / \'bundle_checksums.json\').read_text(encoding=\'utf-8\'))\n    for relative, expected in manifest[\'files\'].items():\n        if _sha(root / relative) != expected:\n            raise ValueError(f\'Bundle checksum mismatch: {relative}\')\n    metadata = json.loads((root / \'bundle_metadata.json\').read_text(encoding=\'utf-8\'))\n    for name in [\'numpy\', \'pandas\', \'scipy\', \'scikit-learn\', \'xgboost\', \'joblib\']:\n        if importlib.metadata.version(name) != metadata[\'versions\'][name]:\n            raise RuntimeError(f\'Runtime version mismatch: {name}. Use requirements_runtime.txt.\')\n    if platform.python_version().split(\'.\')[:2] != metadata[\'versions\'][\'python\'].split(\'.\')[:2]:\n        raise RuntimeError(\'Python major/minor mismatch.\')\n    model = joblib.load(root / \'pipeline.joblib\')\n    check_is_fitted(model)\n    if list(model.features) != metadata[\'raw_features\'] or list(model.class_names) != metadata[\'class_names\']:\n        raise ValueError(\'Serialized schema differs from bundle metadata.\')\n    return model\n\n\ndef predict_csv(bundle_dir, csv_path, output_path, chunksize=1024):\n    model = load_bundle(bundle_dir)\n    output = Path(output_path)\n    first = True\n    for frame in pd.read_csv(csv_path, chunksize=chunksize):\n        probabilities = model.predict_proba_raw(frame)\n        prediction = probabilities.argmax(axis=1)\n        result = pd.DataFrame({\'predicted_id\': prediction,\n                               \'predicted_label\': [model.class_names[i] for i in prediction]})\n        for i, label in enumerate(model.class_names):\n            result[f\'p_{label}\'] = probabilities[:, i]\n        result.to_csv(output, mode=\'w\' if first else \'a\', header=first, index=False)\n        first = False\n    return output\n'
RUNTIME_DIR = OUT / 'runtime'
RUNTIME_DIR.mkdir(exist_ok=True)
module_path = RUNTIME_DIR / 'blending_module.py'
if module_path.exists():
    assert module_path.read_text(encoding='utf-8') == RUNTIME_MODULE
else:
    module_path.write_text(RUNTIME_MODULE, encoding='utf-8')
# Không cho cache import module của run khác trong cùng kernel.
if 'blending_module' in sys.modules:
    previous_module = sys.modules['blending_module']
    assert Path(previous_module.__file__).resolve() == module_path.resolve(), 'Restart kernel trước khi đổi run.'
else:
    sys.path.insert(0, str(RUNTIME_DIR))
blending_module = importlib.import_module('blending_module')
BlendingClassifier = blending_module.BlendingClassifier
BLENDING = BlendingClassifier(BASE_PIPELINES, META_MODEL, FEATURES, CLASSES).initialize_pretrained()
check_is_fitted(BLENDING)


In [ ]:
def benchmark_raw(model, frame, repeats=BENCHMARK_REPEATS):
    records = []
    for size in [1, min(1024, len(frame))]:
        batch = frame.iloc[:size].copy()
        model.predict_proba(batch)  # warm-up, không tính vào repeats
        durations = []
        with PeakRSS() as memory, threadpool_limits(limits=N_THREADS):
            for _ in range(repeats):
                started = time.perf_counter()
                model.predict_proba(batch)
                durations.append(time.perf_counter() - started)
        records.append({'batch_size': size, 'repeats': repeats,
            'median_seconds': float(np.median(durations)), 'p95_seconds': float(np.quantile(durations, 0.95)),
            'median_ms_per_row': float(np.median(durations) * 1000 / size),
            'throughput_rows_per_second': float(size / np.median(durations)),
            'baseline_rss_mib': memory.baseline, 'peak_rss_mib': memory.peak,
            'scope': 'DataFrame raw API; fixed preprocessing + base3 + concat + meta (CSV parsing excluded)'})
    return records


HANDOFF = OUT / 'handoff'
if (HANDOFF / 'bundle_checksums.json').exists():
    verify_seal(HANDOFF, 'bundle_checksums.json')
    LOADED = blending_module.load_bundle(HANDOFF)
else:
    assert not SEARCH_LOCKED
    HANDOFF.mkdir(exist_ok=True)
    (HANDOFF / 'blending_module.py').write_text(RUNTIME_MODULE, encoding='utf-8')
    for name in BASE_ORDER:
        destination = HANDOFF / 'base_models' / name
        destination.mkdir(parents=True, exist_ok=True)
        for filename in ['pipeline.joblib', 'model.joblib', 'config.json', 'bundle_metadata.json',
                         'feature_schema.json', 'preprocessing_config.json', 'label_mapping.json']:
            shutil.copy2(baseline_file('handoff/' + name + '/' + filename), destination / filename)
    shutil.copy2(FINAL_DIR / 'meta_model.joblib', HANDOFF / 'meta_model.joblib')
    shutil.copy2(FINAL_DIR / 'fit_summary.json', HANDOFF / 'meta_fit_summary.json')
    shutil.copy2(PROCESSED_DIR / 'preprocessing_config.json', HANDOFF / 'preprocessing_config.json')
    shutil.copy2(OUT / 'meta_feature_schema.json', HANDOFF / 'meta_feature_schema.json')
    atomic_json(HANDOFF / 'feature_schema.json', {'raw_features': FEATURES, 'n_raw_features': 44,
        'dtype': 'float64 frame -> fixed per-base preprocessing -> float32',
        'extra_columns': 'ignored; never used as model inputs', 'missing_features': 'error',
        'missing_values': 'NaN/Inf handled by saved train medians', 'numeric_parse_errors': 'error'})
    atomic_json(HANDOFF / 'label_mapping.json', {'class_names': CLASSES, 'class_to_id': {c: i for i, c in enumerate(CLASSES)}})
    atomic_json(HANDOFF / 'config.json', {'protocol_hash': PROTOCOL_HASH, 'base_order': BASE_ORDER,
        'base_snapshot': BASE_SNAPSHOT, 'meta_spec': SELECTED['spec'], 'meta_training_rows': len(Y_META),
        'meta_training_support': {c: int((Y_META == i).sum()) for i, c in enumerate(CLASSES)},
        'decision': 'argmax', 'base_refit_train_plus_val': False})
    atomic_joblib(HANDOFF / 'pipeline.joblib', BLENDING)
    LOADED = joblib.load(HANDOFF / 'pipeline.joblib')
    check_is_fitted(LOADED)
    rng = np.random.default_rng(SEED)
    golden_idx = np.unique(np.concatenate([rng.choice(np.flatnonzero(Y_META == i), min(8, (Y_META == i).sum()), replace=False)
                                          for i in range(8)]))
    golden_frame = pd.DataFrame(np.asarray(XVAL_RAW[VAL_KEPT[golden_idx]], dtype=np.float64), columns=FEATURES)
    manual_z = np.concatenate([BASE_PIPELINES[n].predict_proba(golden_frame) for n in BASE_ORDER], axis=1).astype(np.float32)
    np.testing.assert_allclose(manual_z, ZVAL[golden_idx], rtol=1e-5, atol=1e-6)
    manual_p = META_MODEL.predict_proba(manual_z)
    np.testing.assert_allclose(LOADED.predict_proba_raw(golden_frame), manual_p, rtol=1e-5, atol=1e-6)
    assert np.array_equal(LOADED.predict_raw(golden_frame), manual_p.argmax(axis=1))
    assert np.array_equal(BLENDING.predict_raw(golden_frame), LOADED.predict_raw(golden_frame))
    # Schema order, extra labels, and recorded missing-value handling.
    reordered = golden_frame[FEATURES[::-1]].assign(label='ignored')
    assert np.array_equal(LOADED.predict_raw(reordered), LOADED.predict_raw(golden_frame))
    missing_probe = golden_frame.iloc[:2].copy()
    missing_probe.iloc[0, 0] = np.nan
    missing_probe.iloc[1, 1] = np.inf
    reference_missing = missing_probe.replace([np.inf, -np.inf], np.nan)
    expected_missing = META_MODEL.predict_proba(np.concatenate(
        [BASE_PIPELINES[n].predict_proba(reference_missing) for n in BASE_ORDER], axis=1).astype(np.float32))
    np.testing.assert_allclose(LOADED.predict_proba_raw(missing_probe), expected_missing, rtol=1e-5, atol=1e-6)
    with (HANDOFF / 'golden_samples.npz').open('wb') as stream:
        np.savez_compressed(stream, X_raw=golden_frame.to_numpy(), meta_features=manual_z, probabilities=manual_p,
                            predictions=manual_p.argmax(axis=1), y_true=Y_META[golden_idx],
                            evaluation_val_indices=golden_idx, processed_val_indices=VAL_KEPT[golden_idx])
    golden_frame.to_csv(HANDOFF / 'example_raw_features.csv', index=False, float_format='%.17g')
    csv_reload = pd.read_csv(HANDOFF / 'example_raw_features.csv', float_precision='round_trip')
    np.testing.assert_allclose(LOADED.predict_proba_raw(csv_reload), manual_p, rtol=1e-5, atol=1e-6)
    assert np.array_equal(LOADED.predict_raw(csv_reload), manual_p.argmax(axis=1))
    # 1024 val rows chỉ dùng benchmark; không dùng benchmark để thay threshold/config.
    benchmark_idx = np.unique(np.linspace(0, len(Y_META) - 1, min(1024, len(Y_META)), dtype=np.int64))
    benchmark_frame = pd.DataFrame(np.asarray(XVAL_RAW[VAL_KEPT[benchmark_idx]], dtype=np.float64), columns=FEATURES)
    timing = []
    for name in BASE_ORDER:
        timing.extend([{'model': name, **record} for record in benchmark_raw(BASE_PIPELINES[name], benchmark_frame)])
    timing.extend([{'model': 'Blending', **record} for record in benchmark_raw(LOADED, benchmark_frame)])
    pd.DataFrame(timing).to_csv(HANDOFF / 'inference_benchmark.csv', index=False)
    (HANDOFF / 'requirements_runtime.txt').write_text(
        '\n'.join(f'{k}=={VERSIONS[k]}' for k in ['numpy', 'pandas', 'scipy', 'scikit-learn', 'xgboost', 'joblib']) + '\n', encoding='utf-8')
    atomic_json(HANDOFF / 'bundle_metadata.json', {'run_id': RUN_ID, 'protocol_hash': PROTOCOL_HASH,
        'author_branch': AUTHOR_BRANCH, 'template_source_sha256': TEMPLATE_SOURCE_SHA256,
        'raw_features': FEATURES, 'meta_columns': META_COLUMNS, 'class_names': CLASSES,
        'versions': VERSIONS, 'python_major_minor': '.'.join(VERSIONS['python'].split('.')[:2]),
        'base_snapshot': BASE_SNAPSHOT, 'meta_spec': SELECTED['spec'],
        'preprocessing_fit_scope': PROTOCOL['preprocessing_fit_scope'],
        'meta_training_rows': len(Y_META), 'history_disclosure': PROTOCOL['history_disclosure'],
        'save_load_golden_parity': True, 'csv_round_trip_parity': True,
        'purpose': 'frozen inference bundle; model selection must not use future test metrics'})
    (HANDOFF / 'README_HANDOFF.md').write_text(
        '# Bàn giao holdout blending 8 lớp\n\n'
        'Model nguồn nhánh duong. Cài requirements_runtime.txt trên Python ' + '.'.join(VERSIONS['python'].split('.')[:2]) + '.\n'
        'Chỉ load bundle đáng tin cậy do nhóm tạo; joblib thực thi Python khi load.\n\n'
        '```python\nimport sys\nfrom pathlib import Path\nimport pandas as pd\n'
        'bundle = Path("/path/to/handoff")\nsys.path.insert(0, str(bundle))\n'
        'from blending_module import load_bundle\nmodel = load_bundle(bundle)\n'
        'frame = pd.read_csv(bundle / "example_raw_features.csv", float_precision="round_trip")\n'
        'labels = model.predict_raw(frame)\nprobabilities = model.predict_proba_raw(frame)\n'
        'Z = model.build_meta_features(frame)\n```\n\n'
        'CSV có đủ 44 feature đúng schema; extra columns bỏ qua, label không là đầu vào. '
        'Giữ Magnitue đúng chính tả schema. Input này là feature table, không phải PCAP.\n\n'
        'Base dùng pilot 1M; preprocessing fit full source train; meta fit val sau mask. '
        'Val/test đã từng được xem. Điểm OOF là development, không phải xác nhận độc lập.\n\n'
        'Kiên: demo cần bundle và mẫu query, không cần train/test. Đo RAM/latency trên máy triển khai.\n'
        'Hải: LIME/CF raw 44 feature gọi model.predict_proba_raw cho MỌI query perturbation; '
        'meta-level 24 score là mức giải thích khác. Không giữ Z cũ khi thay raw query.\n'
        'pipeline.joblib chứa cả hệ thống; base_models/ và meta_model.joblib là bản sao cho audit/XAI riêng.\n',
        encoding='utf-8')
    seal(HANDOFF, 'bundle_checksums.json', {'protocol_hash': PROTOCOL_HASH})
    LOADED = blending_module.load_bundle(HANDOFF)
# Xác nhận loader công khai, ngay cả ở resume.
with np.load(HANDOFF / 'golden_samples.npz', allow_pickle=False) as g:
    reloaded_frame = pd.DataFrame(g['X_raw'], columns=FEATURES)
    np.testing.assert_allclose(LOADED.predict_proba_raw(reloaded_frame), g['probabilities'], rtol=1e-5, atol=1e-6)
    assert np.array_equal(LOADED.predict_raw(reloaded_frame), g['predictions'])
print('Handoff + public loader + golden/CSV parity: PASS')
display(pd.read_csv(HANDOFF / 'inference_benchmark.csv'))


## 12. Freeze trước test

Freeze khóa protocol, trial/meta, bundle checksums và model. Lần resume sau chỉ đánh giá lại
những phần chưa hoàn tất; không mở search mới sau freeze. Không chỉnh threshold sau xem test.


In [ ]:
FREEZE_PATH = OUT / 'frozen_blending.json'
FREEZE_CONTENT = {'run_id': RUN_ID, 'protocol_hash': PROTOCOL_HASH,
    'protocol_file_sha256': sha256(OUT / 'protocol.json'),
    'selected_meta_sha256': sha256(OUT / 'selected_meta.json'),
    'meta_model_sha256': sha256(FINAL_DIR / 'meta_model.joblib'),
    'handoff_checksums_sha256': sha256(HANDOFF / 'bundle_checksums.json'),
    'pipeline_sha256': sha256(HANDOFF / 'pipeline.joblib'), 'selected_trial': SELECTED['trial_id'],
    'meta_spec': SELECTED['spec'], 'decision': 'argmax', 'test_interpretation': 'historically viewed test'}
if FREEZE_PATH.exists():
    frozen = read_json(FREEZE_PATH)
    assert {k: v for k, v in frozen.items() if k != 'frozen_at'} == FREEZE_CONTENT
else:
    atomic_json(FREEZE_PATH, {**FREEZE_CONTENT, 'frozen_at': now()})
SEARCH_LOCKED = True


def verify_frozen():
    frozen = read_json(FREEZE_PATH)
    assert frozen['protocol_hash'] == PROTOCOL_HASH and frozen['meta_spec'] == SELECTED['spec']
    require_hash(OUT / 'protocol.json', frozen['protocol_file_sha256'])
    require_hash(OUT / 'selected_meta.json', frozen['selected_meta_sha256'])
    require_hash(FINAL_DIR / 'meta_model.joblib', frozen['meta_model_sha256'])
    require_hash(HANDOFF / 'bundle_checksums.json', frozen['handoff_checksums_sha256'])
    require_hash(HANDOFF / 'pipeline.joblib', frozen['pipeline_sha256'])
    verify_seal(HANDOFF, 'bundle_checksums.json')
    return frozen


verify_frozen()
atomic_json(OUT / 'run_status.json', {'run_id': RUN_ID, 'protocol_hash': PROTOCOL_HASH,
    'stage': 'frozen_ready_for_test', 'test_completed': (OUT / 'test/complete.json').exists(),
    'at': now(), 'next': 'Review CV; resume same run with RUN_FINAL_TEST=True for historical final test'})
print('FROZEN:', FREEZE_PATH)
if not RUN_FINAL_TEST:
    print('DỪNG TRƯỚC TEST. Save Version giữ output. Đọc development_comparison.csv và meta_cv_results.csv.')
    print('Lần sau đặt RESUME_FROM tới thư mục run này và RUN_FINAL_TEST=True.')


## 13. Test lịch sử — chỉ khi RUN_FINAL_TEST=True

So sánh 5 pipeline trên **cùng mask/row IDs**. Base/soft vote dùng caches đã kiểm tra;
blending gọi lại toàn pipeline theo batch để đo chi phí thật, đồng thời kiểm tra parity với Z_test.
Checkpoint sau mỗi batch. Khi hoàn tất có test completion marker; resume chỉ xác nhận và đọc báo cáo,
không chọn/fit thêm từ điểm test. Mẫu test chưa được đọc ở các bước tìm kiếm trên.


In [ ]:
TEST_DIR = OUT / 'test'
if not RUN_FINAL_TEST:
    print('Test skipped. RUN_FINAL_TEST=False; chưa tạo Z_test hay mở test labels.')
elif (TEST_DIR / 'complete.json').exists():
    verify_frozen()
    verify_seal(TEST_DIR)
    print('Test đã hoàn tất trong run này. Đọc lại báo cáo đã lưu, không đánh giá/chọn lại.')
    display(pd.read_csv(TEST_DIR / 'final_comparison.csv'))
else:
    frozen = verify_frozen()
    XTEST_RAW, TEST_KEPT, Y_TEST, TEST_ROWS, ZTEST = prepare_split('test')
    TEST_DIR.mkdir(exist_ok=True)
    test_identity = {'protocol_hash': PROTOCOL_HASH, 'freeze_sha256': sha256(FREEZE_PATH),
        'rows': len(Y_TEST), 'z_sha256': sha256(OUT / 'test_data/Z_test.npy'),
        'test_mask_sha256': BASE_CHECKSUMS['test_float32_kept_processed_indices.npy']}
    blend_folder = TEST_DIR / 'Blending'
    blend_folder.mkdir(exist_ok=True)
    p_path = blend_folder / 'probabilities.npy'
    progress_path = TEST_DIR / 'prediction_progress.json'
    if progress_path.exists():
        progress = read_json(progress_path)
        assert progress['identity'] == test_identity
        p_blend = np.lib.format.open_memmap(p_path, mode='r+')
    else:
        p_blend = np.lib.format.open_memmap(p_path, mode='w+', dtype=np.float32, shape=(len(Y_TEST), 8))
        progress = {'identity': test_identity, 'next_row': 0, 'batch_timings': []}
        atomic_json(progress_path, progress)
    assert p_blend.shape == (len(Y_TEST), 8)
    for start in range(progress['next_row'], len(Y_TEST), BATCH_SIZE):
        stop = min(start + BATCH_SIZE, len(Y_TEST))
        with threadpool_limits(limits=N_THREADS), PeakRSS() as memory:
            began = time.perf_counter()
            frame = pd.DataFrame(np.asarray(XTEST_RAW[TEST_KEPT[start:stop]], dtype=np.float64), columns=FEATURES)
            p = LOADED.predict_proba_raw(frame)
            seconds = time.perf_counter() - began
        expected = META_MODEL.predict_proba(np.asarray(ZTEST[start:stop], dtype=np.float32))
        np.testing.assert_allclose(p, expected, rtol=1e-5, atol=1e-6)
        assert np.array_equal(p.argmax(axis=1), expected.argmax(axis=1))
        check_probabilities(p, stop-start)
        p_blend[start:stop] = p
        p_blend.flush()
        progress['next_row'] = stop
        progress['batch_timings'].append({'start': start, 'stop': stop, 'seconds': seconds,
            'baseline_rss_mib': memory.baseline, 'peak_rss_mib': memory.peak})
        atomic_json(progress_path, progress)
        print('Full pipeline test:', stop, '/', len(Y_TEST), flush=True)
    check_probabilities(p_blend, len(Y_TEST))
    atomic_json(blend_folder / 'inference_full_test.json', {
        'seconds': sum(r['seconds'] for r in progress['batch_timings']),
        'rows': len(Y_TEST), 'batch_size': BATCH_SIZE,
        'throughput_rows_per_second': len(Y_TEST) / sum(r['seconds'] for r in progress['batch_timings']),
        'peak_rss_mib': max(r['peak_rss_mib'] for r in progress['batch_timings']),
        'scope': 'raw array selection + DataFrame + all 3 fixed preprocessors/base models + concat + meta',
        'excluded': 'checkpoint/hash/export time; CSV parsing; cold model loading',
        'all_rows_parity_with_cached_Z_test': True, 'batch_timings': progress['batch_timings']})
    final_rows = []
    for i, name in enumerate(BASE_ORDER):
        stats = export_metrics(TEST_DIR / name, Y_TEST, ZTEST[:, i*8:(i+1)*8], name + ' — historical test')
        final_rows.append({'model': name, 'inference_mode': 'verified baseline probability cache',
            'base_fit_seconds': BASE_SNAPSHOT[name]['original_fit_seconds'], 'meta_fit_seconds': 0.0, **stats})
    soft_test = np.mean(np.asarray(ZTEST).reshape(-1, 3, 8), axis=1, dtype=np.float32)
    stats = export_metrics(TEST_DIR / 'SoftVoting', Y_TEST, soft_test, 'Equal soft voting — historical test')
    final_rows.append({'model': 'SoftVoting', 'inference_mode': 'verified base caches + equal average',
        'base_fit_seconds': sum(BASE_SNAPSHOT[n]['original_fit_seconds'] for n in BASE_ORDER),
        'meta_fit_seconds': 0.0, **stats})
    stats = export_metrics(blend_folder, Y_TEST, p_blend, 'Frozen blending — historical test')
    final_rows.append({'model': 'Blending', 'inference_mode': 'full raw pipeline; cache parity all rows',
        'base_fit_seconds': sum(BASE_SNAPSHOT[n]['original_fit_seconds'] for n in BASE_ORDER),
        'meta_fit_seconds': read_json(FINAL_DIR / 'fit_summary.json')['fit_seconds'],
        'meta_search_checkpointed_seconds': LEDGER['checkpointed_seconds'], **stats})
    for i, name in enumerate(BASE_ORDER):
        TEST_ROWS[name + '_prediction'] = ZTEST[:, i*8:(i+1)*8].argmax(axis=1)
    TEST_ROWS['soft_voting_prediction'] = soft_test.argmax(axis=1)
    TEST_ROWS['blending_prediction'] = np.asarray(p_blend).argmax(axis=1)
    TEST_ROWS.to_csv(TEST_DIR / 'predictions.csv.gz', index=False, compression='gzip')
    TEST_ROWS.loc[TEST_ROWS.true_class_id.isin([1, 7])].to_csv(TEST_DIR / 'rare_cases.csv', index=False)
    rare_idx = np.flatnonzero(np.isin(Y_TEST, [1, 7]))
    rare_fingerprints = []
    for raw_row in np.asarray(XTEST_RAW[TEST_KEPT[rare_idx]], dtype='<f4'):
        raw_row[raw_row == 0] = 0.0
        rare_fingerprints.append(hashlib.blake2b(raw_row.tobytes(), digest_size=16).digest())
    _, rare_inverse = np.unique(np.asarray(rare_fingerprints, dtype='V16'), return_inverse=True)
    test_groups = np.full(len(Y_TEST), -1, dtype=np.int64)
    test_groups[rare_idx] = rare_inverse
    pd.DataFrame(rare_recall_ci(Y_TEST, TEST_ROWS.blending_prediction.to_numpy(), test_groups)).to_csv(
        TEST_DIR / 'rare_recall_ci.csv', index=False)
    final_table = pd.DataFrame(final_rows)
    final_table.to_csv(TEST_DIR / 'final_comparison.csv', index=False)
    atomic_json(TEST_DIR / 'evaluation_protocol.json', {**test_identity,
        'test_interpretation': 'historically viewed test; no independent confirmation claim',
        'meta_fit_scope': 'all masked val; base fixed pilot 1M; preprocessing full source train',
        'selection': 'frozen before test; no refit/threshold search on test',
        'data_budget_caveat': 'Blending uses val labels in addition to base train; not equal labeled-data budget',
        'negative_result_policy': 'Report all outcomes; no replacement candidate selected from test'})
    del p_blend
    progress_path.unlink()
    seal(TEST_DIR, extra={'identity': test_identity})
    del soft_test, test_groups
    gc.collect()
    display(final_table[['model', 'macro_f1', 'precision_BruteForce', 'recall_BruteForce',
                        'precision_Web-Based', 'recall_Web-Based', 'accuracy']])
# Root summary có thể dựng lại từ test đã seal nếu kernel dừng ngay sau marker.
if RUN_FINAL_TEST and (TEST_DIR / 'complete.json').exists():
    verify_seal(TEST_DIR)
    shutil.copy2(TEST_DIR / 'final_comparison.csv', OUT / 'final_comparison.csv')
    atomic_json(OUT / 'run_status.json', {'run_id': RUN_ID, 'protocol_hash': PROTOCOL_HASH,
        'stage': 'historical_test_completed', 'test_completed': True, 'at': now(),
        'next': 'Report metrics and tradeoffs; use frozen bundle for XAI/demo; do not tune against test'})


## 14. Đầu ra, cách đọc kết quả và bàn giao

Output: `/kaggle/working/blending_experiments/<run_id>/`.

- `protocol.json`, `input_integrity.json`, `base_models_snapshot.json`: nguồn/run/hash và lịch sử đánh giá.
- `meta_data/`: Z_val float32, y_meta, row IDs và alignment audit; `folds/`: groups, fold IDs/support.
- `trials/`: fold checkpoints, OOF metrics/confusion/timing/model của từng config.
- `meta_cv_results.csv`, `development_comparison.csv`, `development_rare_cases.csv`: lựa chọn và chẩn đoán.
- `selected_meta.json`, `final_meta/`, `frozen_blending.json`: candidate khóa trước test.
- `handoff/`: bundle suy luận, source module, requirements, config/schema, golden, CSV và README.
- Sau khi bật test: `test_data/`, `test/`, `final_comparison.csv`, per-class reports, confusion matrices,
  rare cases/CI và full-pipeline inference time. Peak RSS bao gồm các model khác còn ở kernel;
  phải đo lại RAM trên máy demo để kết luận phù hợp local 8 GB.

Đọc **macro-F1 cùng precision/recall/F1 của cả 8 lớp**, đặc biệt BF/Web, Normal false alarms và thời gian.
Accuracy một mình dễ che khuất lỗi lớp hiếm. OOF chọn cấu hình không là bằng chứng xác nhận độc lập;
không thay model/threshold theo final test. Nếu không tăng điểm, báo error coverage và kết quả âm.
Các đối chứng base có ít nhãn training hơn blending: muốn tách lợi ích kiến trúc khỏi lợi ích dữ liệu,
làm một ablation protocol riêng; không refit base bên trong bundle blending hiện tại.

Hải: XAI meta-level dùng 24 score (không phải traffic features); XAI toàn pipeline dùng 44 raw feature
và gọi lại cả pipeline cho mỗi perturbation. Kiên: dùng API trong README_HANDOFF, input feature CSV đủ
44 cột; demo không cần tải toàn train/test. Notebook không tự tick checklist nghiệm thu hay commit/push.


In [ ]:
print('Output:', OUT)
print('Status:', read_json(OUT / 'run_status.json'))
print('Training/search:', LEDGER['checkpointed_seconds'], 'seconds checkpointed search;',
      read_json(FINAL_DIR / 'fit_summary.json')['fit_seconds'], 'seconds final meta fit')
if (OUT / 'final_comparison.csv').exists():
    display(pd.read_csv(OUT / 'final_comparison.csv'))
else:
    print('Chưa có final test. Save Version output; resume cùng protocol và bật RUN_FINAL_TEST khi sẵn sàng.')


## References

1. [Plan nhánh duong](../work_dir/Nguyen_Ngoc_Duong_04_ke_hoach_blending_meta_model.md).
2. [sklearn StackingClassifier](https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.StackingClassifier.html):
   phân biệt prefit base-models và final estimator; notebook tạo Z thủ công cho holdout blending.
3. [StratifiedGroupKFold](https://scikit-learn.org/1.6/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html):
   giữ groups giữa các fold, stratification không bảo đảm support từng lớp bằng nhau.
4. [Common pitfalls](https://scikit-learn.org/1.6/common_pitfalls.html): fit statistics trong training, tránh leakage.
5. [XGBoost parameters](https://xgboost.readthedocs.io/en/stable/parameter.html): shallow multiclass trees/regularization.
6. [Developing sklearn estimators](https://scikit-learn.org/1.6/developers/develop.html): estimator API/fitted-state.
7. [Toward Enhanced Attack Detection and Explanation…](https://doi.org/10.1109/ACCESS.2023.3336678):
   tham khảo ensemble/XAI; phương án DT/RF/XGBoost + 24-score meta này không là tái lập nguyên trạng paper.
